[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/claude-api-certified/notebooks/day-09-agents.ipynb#scrollTo=aa10bb20)

---
# Day 9 · Agents & Claude Code Workflows
**certified-journeys / claude-api-certified** · Agent loops, error budgets & multi-agent patterns

> **Goal for today:** Build a production-grade agent with proper ReAct loops, error budgets, structured logging, and a multi-agent orchestrator/specialist pattern — the foundations of any serious Claude-powered automation.

In [ ]:
%pip install -q anthropic

In [ ]:
import os
import json
import time
import logging
from dataclasses import dataclass, field
from typing import Any
import anthropic

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(name)s] %(message)s")

## Step 1 · The ReAct agent pattern

ReAct = **Re**asoning + **Act**ing. The agent observes the environment, thinks about what to do, acts (calls a tool), observes the result, and repeats until done.

```
OBSERVE (user input + history)
   ↓
THINK (Claude generates next step)
   ↓
ACT (call a tool if needed)
   ↓
OBSERVE (tool result added to history)
   ↓
REPEAT until stop_reason == end_turn
```

In [ ]:
@dataclass
class AgentStep:
    """A single step in an agent's execution trace."""
    step: int
    type: str  # 'think', 'tool_call', 'tool_result', 'final'
    content: Any
    timestamp: float = field(default_factory=time.time)


@dataclass
class AgentResult:
    """Result of a complete agent run."""
    success: bool
    answer: str
    steps: list[AgentStep]
    iterations: int
    halt_reason: str  # 'end_turn', 'max_iterations', 'error', 'budget_exceeded'


# Research tools — mock implementations
AGENT_TOOLS = [
    {
        "name": "search_web",
        "description": "Search for information on a topic. Returns a list of relevant facts.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "read_page",
        "description": "Read the full content of a web page given its URL.",
        "input_schema": {
            "type": "object",
            "properties": {
                "url": {"type": "string", "description": "The URL to read"}
            },
            "required": ["url"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a local file.",
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {"type": "string"},
                "content": {"type": "string"}
            },
            "required": ["filename", "content"]
        }
    }
]


MOCK_SEARCH_DB = {
    "anthropic claude api": ["Claude API uses Messages endpoint", "Supports tool use and streaming", "Models: Haiku, Sonnet, Opus"],
    "prompt caching": ["Cache lives for 5 minutes", "Min 1024 tokens to cache", "90% cost reduction on cache hit"],
    "mcp protocol": ["Open standard for AI tool integration", "Supports tools, resources, prompts", "Created by Anthropic in 2024"],
}


def execute_tool(name: str, inputs: dict) -> str:
    """Mock tool executor."""
    if name == "search_web":
        query = inputs.get("query", "").lower()
        for key, facts in MOCK_SEARCH_DB.items():
            if any(word in query for word in key.split()):
                return json.dumps({"results": facts, "query": query})
        return json.dumps({"results": ["No results found"], "query": query})

    elif name == "read_page":
        url = inputs.get("url", "")
        return json.dumps({"url": url, "content": f"[Mock page content for {url}]"})

    elif name == "write_file":
        filename = inputs.get("filename", "output.txt")
        content = inputs.get("content", "")
        with open(filename, "w") as f:
            f.write(content)
        return json.dumps({"written": True, "filename": filename, "chars": len(content)})

    return json.dumps({"error": f"Unknown tool: {name}"})


print("Agent tools defined:", [t["name"] for t in AGENT_TOOLS])

**What just happened?**
- `AgentStep` and `AgentResult` give you a structured audit trail — every tool call is recorded.
- **Structured execution traces** are essential for debugging agents in production.
- `halt_reason` distinguishes clean completion from timeout/error — different states need different handling.
- Mock tools let you build and test the agent loop before integrating real APIs.

## Step 2 · Build the agent with an error budget

An error budget defines how many failures the agent can tolerate before halting. Without it, a single broken tool creates an infinite loop.

In [ ]:
class ReActAgent:
    def __init__(
        self,
        tools: list[dict],
        system: str = "",
        max_iterations: int = 10,
        max_errors: int = 3,  # error budget
        model: str = MODEL
    ):
        self.tools = tools
        self.system = system
        self.max_iterations = max_iterations
        self.max_errors = max_errors
        self.model = model
        self.logger = logging.getLogger(self.__class__.__name__)

    def run(self, task: str) -> AgentResult:
        messages = [{"role": "user", "content": task}]
        steps = []
        errors = 0

        for i in range(self.max_iterations):
            self.logger.info(f"Iteration {i+1}/{self.max_iterations} | Errors: {errors}/{self.max_errors}")

            try:
                kwargs = {"model": self.model, "max_tokens": 1024, "tools": self.tools, "messages": messages}
                if self.system:
                    kwargs["system"] = self.system

                r = client.messages.create(**kwargs)
            except anthropic.APIError as e:
                errors += 1
                self.logger.error(f"API error: {e}")
                if errors >= self.max_errors:
                    return AgentResult(False, "", steps, i+1, "budget_exceeded")
                continue

            messages.append({"role": "assistant", "content": r.content})

            if r.stop_reason == "end_turn":
                final_text = " ".join(b.text for b in r.content if b.type == "text")
                steps.append(AgentStep(i, "final", final_text))
                return AgentResult(True, final_text, steps, i+1, "end_turn")

            if r.stop_reason == "tool_use":
                tool_results = []
                for block in r.content:
                    if block.type == "tool_use":
                        self.logger.info(f"Tool call: {block.name}({json.dumps(block.input)[:100]})")
                        steps.append(AgentStep(i, "tool_call", {"tool": block.name, "input": block.input}))
                        try:
                            result = execute_tool(block.name, block.input)
                            self.logger.info(f"Tool result: {result[:100]}")
                        except Exception as e:
                            result = json.dumps({"error": str(e)})
                            errors += 1
                            self.logger.warning(f"Tool error (budget: {errors}/{self.max_errors}): {e}")
                        steps.append(AgentStep(i, "tool_result", result))
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result
                        })
                        if errors >= self.max_errors:
                            return AgentResult(False, "", steps, i+1, "budget_exceeded")

                messages.append({"role": "user", "content": tool_results})

        return AgentResult(False, "", steps, self.max_iterations, "max_iterations")


# Run the agent
agent = ReActAgent(
    tools=AGENT_TOOLS,
    system="You are a research assistant. Search for information, then write a summary to a file called 'summary.txt'. Be concise.",
    max_iterations=8,
    max_errors=2
)

result = agent.run("Research the Anthropic Claude API and MCP protocol, then write a 3-bullet summary to summary.txt")

print(f"\nAgent result:")
print(f"  Success: {result.success}")
print(f"  Iterations: {result.iterations}")
print(f"  Halt reason: {result.halt_reason}")
print(f"  Steps taken: {len(result.steps)}")
print(f"\nFinal answer: {result.answer[:300]}")

**What just happened?**
- **`max_errors`** is the error budget — after 2 tool failures, the agent stops rather than spiraling.
- **Structured steps** give you a full audit trail of every tool call and result.
- API errors (rate limits, timeouts) are caught separately from tool errors — different retry logic applies.
- **`halt_reason`** tells your application how to respond: retry, escalate to human, or fail gracefully.

## Step 3 · Multi-agent: orchestrator + specialist pattern

Complex tasks need multiple specialized agents. An orchestrator breaks down the task; specialists execute subtasks in parallel or sequence.

In [ ]:
def specialist_agent(role: str, task: str, context: str = "") -> str:
    """A focused single-turn specialist agent."""
    system = f"You are a specialist {role}. Complete only the task given. Be concise and precise."
    if context:
        system += f"\n\nContext from orchestrator: {context}"

    r = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=system,
        messages=[{"role": "user", "content": task}]
    )
    return r.content[0].text


def orchestrator(high_level_task: str) -> dict[str, str]:
    """Break down a task and delegate to specialist agents."""
    logger = logging.getLogger("Orchestrator")

    # Step 1: Orchestrator plans
    plan_prompt = f"""Break this task into exactly 3 subtasks for different specialists.
Respond as JSON: {{"subtasks": [{{"specialist": "role", "task": "what to do"}}]}}

Task: {high_level_task}"""

    r = client.messages.create(
        model=MODEL, max_tokens=512, temperature=0,
        messages=[{"role": "user", "content": plan_prompt}]
    )

    try:
        plan = json.loads(r.content[0].text)
        subtasks = plan.get("subtasks", [])
    except json.JSONDecodeError:
        logger.error("Orchestrator failed to produce valid JSON plan")
        return {}

    logger.info(f"Plan: {len(subtasks)} subtasks")

    # Step 2: Delegate to specialists
    results = {}
    for subtask in subtasks:
        specialist = subtask.get("specialist", "generalist")
        task = subtask.get("task", "")
        logger.info(f"Delegating to {specialist}: {task[:60]}")
        results[specialist] = specialist_agent(specialist, task)

    # Step 3: Synthesize
    synthesis_context = "\n".join(f"[{k}]: {v[:200]}" for k, v in results.items())
    synthesis = specialist_agent(
        role="synthesizer",
        task=f"Combine the specialist outputs into a coherent final answer for: {high_level_task}",
        context=synthesis_context
    )
    results["_synthesis"] = synthesis

    return results


output = orchestrator("Explain what makes the Claude API unique compared to other LLM APIs")
print("Synthesis:")
print(output.get("_synthesis", ""))

**What just happened?**
- The **orchestrator** does planning and coordination; **specialists** do focused execution.
- Each specialist gets a narrow, well-defined task — better than asking one agent to do everything.
- **Context injection** (`context=synthesis_context`) passes partial results between agents.
- For real parallel execution, use `asyncio.gather()` to run specialists concurrently.

## Step 4 · Add observability — log every tool call

Production agents need metrics on every tool call: latency, success rate, and error types.

In [ ]:
from collections import defaultdict

class AgentMetrics:
    def __init__(self):
        self.calls = defaultdict(int)
        self.errors = defaultdict(int)
        self.latencies = defaultdict(list)

    def record(self, tool_name: str, latency: float, success: bool):
        self.calls[tool_name] += 1
        self.latencies[tool_name].append(latency)
        if not success:
            self.errors[tool_name] += 1

    def report(self):
        print("\n=== Agent Metrics ===")
        for tool, count in self.calls.items():
            avg_latency = sum(self.latencies[tool]) / len(self.latencies[tool])
            error_rate = self.errors[tool] / count
            print(f"  {tool}: {count} calls | avg {avg_latency:.3f}s | {error_rate:.0%} error rate")


metrics = AgentMetrics()

def tracked_execute_tool(name: str, inputs: dict) -> str:
    """Wrapper that records metrics for every tool call."""
    start = time.time()
    try:
        result = execute_tool(name, inputs)
        metrics.record(name, time.time() - start, success=True)
        return result
    except Exception as e:
        metrics.record(name, time.time() - start, success=False)
        return json.dumps({"error": str(e)})


# Simulate some tool calls
for _ in range(3):
    tracked_execute_tool("search_web", {"query": "claude api"})
tracked_execute_tool("write_file", {"filename": "test.txt", "content": "hello"})

metrics.report()

**What just happened?**
- **Per-tool metrics** reveal which tools are slow or unreliable — critical for SLA management.
- Error rate per tool tells you where to invest in better error handling or tool redesign.
- In production, send these metrics to your observability stack (Datadog, Grafana, etc.).
- **P99 latency matters more than average** for user-facing agents — track both.

## Step 5 · Agent design principles

A summary of production agent patterns that prevent the most common failures.

In [ ]:
AGENT_DESIGN_PRINCIPLES = {
    "Error budget": "Always set max_errors. A single broken tool should not loop forever.",
    "Iteration cap": "Always set max_iterations. Agents can loop on unexpected inputs.",
    "Structured logging": "Log every tool call with name, input, output, and latency.",
    "Halt reason": "Return why the agent stopped, not just whether it succeeded.",
    "Audit trail": "Record every step so you can replay and debug any run.",
    "Minimal tools": "Give agents only the tools they need — extra tools increase confusion.",
    "Tool validation": "Validate tool inputs before calling. Never trust model-generated strings.",
    "Human handoff": "Define when to escalate to a human — not every failure should retry.",
}

print("Production agent design principles:")
for i, (principle, explanation) in enumerate(AGENT_DESIGN_PRINCIPLES.items(), 1):
    print(f"\n{i}. {principle}")
    print(f"   {explanation}")

**What just happened?**
- These principles come from production failures — each one addresses a common agent breakdown mode.
- **Error budget + iteration cap** are the two most critical safety rails. Never skip them.
- **Human handoff** is often forgotten until an agent causes a production incident.
- **Minimal tools** reduces hallucinated tool calls and narrows the attack surface.

In [ ]:
# Challenge: Add parallel specialist execution
# The current orchestrator runs specialists sequentially.
# Rewrite it to run all specialists in parallel using asyncio.gather().
# This reduces total latency from O(n) to O(1) for n specialists.

import asyncio

async def async_specialist(role: str, task: str) -> tuple[str, str]:
    """Async wrapper for specialist_agent."""
    # Your solution here:
    # Run specialist_agent() in a thread pool (use asyncio.to_thread)
    # Return (role, result) tuple
    pass


async def parallel_orchestrator(high_level_task: str) -> dict:
    """Orchestrator that runs specialists in parallel."""
    # Your solution here:
    # 1. Get the plan (same as before)
    # 2. Use asyncio.gather() to run all specialists at once
    # 3. Collect results and synthesize
    pass


print("Parallel orchestrator skeleton ready — implement above.")
print("Hint: asyncio.gather(*[async_specialist(s, t) for s, t in subtasks])")

---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| ReAct loop | Observe → Think → Act → repeat until end_turn |
| Error budget | `max_errors` — stop after N failures, not at first failure |
| Iteration cap | `max_iterations` — always set it, never omit it |
| Audit trail | `AgentStep` list — record every tool call and result |
| Multi-agent | Orchestrator plans; specialists execute; synthesizer combines |

> **Tip:** Agents need explicit error budgets — decide upfront how many tool failures to tolerate. Silent infinite loops are the #1 production agent failure mode.

---
## What's next
**Day 10** → Capstone: combine all skills into a production-ready, deployed Claude-powered application.

Mark Day 9 complete in your [tracker](../index.html).